# 15 · Combinations: SFT, then DPO and RL (`sft_legal_dpo`, `sft_legal_rl`, `sft_legal_dpo_rl`)

**The question.** Our research question asks which *combination* of post-training methods works best. Notebooks 08–14 found the best setting of each method separately; this notebook combines them:

| Model | Steps | Question it answers |
|---|---|---|
| `sft_legal_dpo` | `sft_legal_moves` → DPO | Does DPO help once the model no longer answers illegally? |
| `sft_legal_rl` | `sft_legal_moves` → RL | Does RL improve move quality once legality is solved? |
| `sft_legal_dpo_rl` | `sft_legal_moves` → DPO → RL | Does RL add to DPO? |

**Why these settings** (the full reasoning is in section 8 of the project notes, "Choosing the combinations"):
- **The SFT model: `sft_legal_moves`** (notebook 13), the best model so far (13.2% of the test puzzles, never illegal). Listing the legal moves also removes the problem that made DPO and RL fail before: both spent much of their training on legality instead of move quality. Half of `dpo_v2`'s rejected answers were illegal moves, and RL's reward pushed `grpo_v1` towards safe pawn moves because any legal move earned a bonus. With the list, the model almost never answers illegally. In GRPO, each answer is scored against the average of the 8 answers to the same position, so when all 8 are legal the legal bonus cancels out and only centipawn loss separates them. And `sft_legal_moves`' weakness (its moves that are not Stockfish's lose as much as random moves) is what RL improved in every earlier run.
- **DPO: `dpo_v3`'s setting** (notebook 09): the standard DPO loss plus the SFT loss on the chosen move, with pairs built from the starting model's own mistakes (chosen = Stockfish's best move, rejected = the model's mistake). It is the only DPO version that did not damage SFT.
- **RL: `grpo_v5`'s setting** (notebook 14): the original reward with the KL penalty β = 0.2, which kept the model closest to SFT, with the fewest pawn moves.

**What stays the same.** Every step uses the legal-move prompt, as `sft_legal_moves` was trained with it. The DPO settings, pair count (4,788) and candidate positions are as in notebook 09; the RL settings and 2,000 positions are as in notebook 14. Each model is graded on the same puzzles with the same grader and greedy decoding. The main comparison is with `sft_legal_moves` itself (the same prompt); `sft_v1`, `dpo_v3` and `grpo_v5` (asked without the list) are shown for reference.

**Running it on Databricks**
1. Open this notebook in the shared Git folder and pick **Serverless GPU → Small (1×A10)**. Plain *Serverless* has no GPU. Medium and Large cost much more and are not needed here.
2. Set `QUICK_CHECK = True` in the Settings cell and click **Run all**. This takes a few minutes, uses tiny sizes, and saves nothing to `results/`. It catches setup problems before the long run. (The first cell restarts Python once; if Run all stops there, click Run all again.)
3. Set `QUICK_CHECK = False` and run the real thing as a **Job**, so the browser can close: New → Job → type *Notebook* → this notebook → compute *Serverless GPU, 1×A10*; add an email notification. Separate Jobs can run at the same time.
4. Afterwards, commit and push **only `results/`** from the Git folder. Leave `notebooks/` unticked: Databricks cannot save notebook outputs to git, so committing a notebook from there would erase its outputs on GitHub. The outputs of a Job run are on the Job's run page; every figure and table is also saved in `results/figures/`.

**Time:** about 30–45 minutes to collect the mistakes, 25 minutes of DPO, 45 minutes for each of the two RL runs, and about 5 minutes of grading per model: about 2.5–3 hours in total (roughly $6 of credits). If the Job stops partway, run it again: models that are already trained and graded are skipped.

**Outputs**
- the pairs in `data/processed/dpo_pairs_sft_legal_mistakes.jsonl` (not on GitHub; reused if the notebook runs again)
- the adapters in `checkpoints/sft_legal_dpo/`, `checkpoints/sft_legal_rl/`, `checkpoints/sft_legal_dpo_rl/` (not on GitHub)
- results: `results/<model>.json`, `results/<model>_dev.json`, `results/<model>_training.json` for each model, and new rows in `results/summary.csv`
- every figure and table of this experiment in `results/figures/combinations/`

In [ ]:
# On Databricks: install our packages and Stockfish for this session. Does nothing on other machines.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # so Databricks finds our code in src/
from chesslm.databricks import setup_databricks

setup_databricks()

In [ ]:
# Settings
QUICK_CHECK = False   # True = a few-minute run with tiny sizes, to check that everything works. Saves nothing to results/.
START = "sft_legal_moves"            # the SFT model every combination starts from (notebook 13)
DPO_RUN = "sft_legal_dpo"            # sft_legal_moves -> DPO
RL_RUN = "sft_legal_rl"              # sft_legal_moves -> RL
DPO_RL_RUN = "sft_legal_dpo_rl"      # sft_legal_moves -> DPO -> RL
DPO_CHANGES = {"loss_type": ["sigmoid", "sft"]}   # dpo_v3's setting: DPO loss + SFT loss on the chosen move
RL_CHANGES = {"beta": 0.2}                        # grpo_v5's setting: KL penalty 0.2, original reward

import copy
import gc
import math

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.width", 200)  # print wide tables on one line

from chesslm import experiments, plots
from chesslm.config import REPO_ROOT, load_config
from chesslm.evaluate import load_eval_model
from chesslm.experiments import (compare_with, figures_dir, grade_puzzles, load_graded, load_training_log, move_kinds,
                                 puzzle_table, reply_function, save_table, save_training)
from chesslm.model import load_tokenizer
from chesslm.preferences import build_mistake_pairs
from chesslm.rewards import to_rl_dataset
from chesslm.train_dpo import split_pairs
from chesslm.train_dpo import train as train_dpo
from chesslm.train_rl import train as train_rl
from chesslm.train_sft import sample_examples

sft_cfg, prompt_cfg, eval_cfg = load_config("sft"), load_config("prompt"), load_config("eval")
prompt_cfg["show_legal_moves"] = True              # every step uses sft_legal_moves' prompt
dpo_cfg, rl_cfg = load_config("dpo"), load_config("rl")
dpo_cfg["rejected_source"] = "sft_mistakes"
dpo_cfg["training"].update(DPO_CHANGES)
rl_cfg["training"].update(RL_CHANGES)
configs = {"model": sft_cfg["model"], "prompt": prompt_cfg, "eval": eval_cfg}
PAIRS_FILE = REPO_ROOT / "data" / "processed" / "dpo_pairs_sft_legal_mistakes.jsonl"
FIGURES_NAME = "combinations"
REDO = False          # True = train and grade again even if a model already has results

if QUICK_CHECK:
    experiments.use_quick_check_folder()
    DPO_RUN, RL_RUN, DPO_RL_RUN = (f"{name}_check" for name in (DPO_RUN, RL_RUN, DPO_RL_RUN))
    PAIRS_FILE = PAIRS_FILE.with_name("dpo_pairs_legal_quick_check.jsonl")
    FIGURES_NAME += "_check"
    REDO = True
    dpo_cfg["target_pairs"] = 24
    dpo_cfg["training"].update(logging_steps=1, eval_steps=1, batch_size=4, effective_batch_size=8, val_fraction=0.25)
    rl_cfg["training"].update(num_positions=8, logging_steps=1)
PUZZLE_LIMIT = 5 if QUICK_CHECK else None
START_ADAPTER = REPO_ROOT / "checkpoints" / START
assert START_ADAPTER.exists(), f"{START_ADAPTER} not found: run notebook 13 first"
FIGURES = figures_dir(FIGURES_NAME)
print(f"Start: {START} | DPO loss {dpo_cfg['training']['loss_type']} | RL beta {rl_cfg['training']['beta']} | "
      f"legal moves in the prompt: {prompt_cfg['show_legal_moves']} | figures and tables -> {FIGURES}")


def adapter_dir(run):
    return REPO_ROOT / "checkpoints" / run


def is_done(run):
    """True if the model is already trained and graded on the test puzzles (then it is not trained again)."""
    return not REDO and (adapter_dir(run) / "adapter_config.json").exists() and (experiments.RESULTS / f"{run}.json").exists()


def grade(run):
    gc.collect()
    torch.cuda.empty_cache()
    tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=adapter_dir(run))
    grade_puzzles(model, tokenizer, "dev", f"{run}_dev", configs, str(adapter_dir(run)), PUZZLE_LIMIT)
    grade_puzzles(model, tokenizer, "test", run, configs, str(adapter_dir(run)), PUZZLE_LIMIT)
    del model
    gc.collect()
    torch.cuda.empty_cache()

## 1. Collect `sft_legal_moves`' own mistakes as DPO pairs

As in notebook 09, with `sft_legal_moves` in place of `sft_v1`. The candidate positions are the same: training positions in the same shuffled order, leaving out the 20,000 positions the SFT model was trained on. The model answers them with the legal-move prompt and greedy decoding; Stockfish checks each answer. Every illegal or unparseable answer, and every legal answer losing at least 200 centipawns, becomes a pair (chosen = Stockfish's best move, rejected = the model's answer). We stop at 4,788 pairs, as in notebooks 05 and 09.

**What to look for.** With the list, almost all mistakes should now be legal blunders rather than illegal moves (for `sft_v1`, half were illegal). The table compares the new pairs with `dpo_v2`'s.

In [ ]:
train_positions = pd.read_parquet(REPO_ROOT / "data" / "processed" / "train.parquet")
sft_positions = sample_examples(train_positions, sft_cfg["data"]["train_examples"], sft_cfg["training"]["seed"])
candidates = train_positions.drop(sft_positions.index).sample(frac=1, random_state=dpo_cfg["seed"])
print(f"{len(candidates):,} candidate positions (training positions minus the {len(sft_positions):,} {START} trained on)")

if PAIRS_FILE.exists() and not QUICK_CHECK:
    pairs = pd.read_json(PAIRS_FILE, lines=True)
    print(f"Loaded {len(pairs):,} pairs from {PAIRS_FILE.name}")
else:
    tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=START_ADAPTER)
    reply = reply_function(model, tokenizer, prompt_cfg, eval_cfg["generation"])
    pairs, answered = build_mistake_pairs(candidates, reply, prompt_cfg, dpo_cfg,
                                          {**eval_cfg["stockfish"], "depth": dpo_cfg["stockfish_depth"]},
                                          batch_positions=64 if QUICK_CHECK else 2000)
    pairs["positions_answered"] = answered
    pairs.to_json(PAIRS_FILE, orient="records", lines=True)
    del model
    gc.collect()
    torch.cuda.empty_cache()
answered = int(pairs.positions_answered.iloc[0])


def pair_summary(p, answered):
    kinds = p.rejected_kind.value_counts(normalize=True) * 100
    legal = p.rejected_cp_loss.dropna()
    return {"Pairs": len(p), "Positions answered": answered,
            "Mistake rate (%)": round(len(p) / answered * 100, 1) if answered else None,
            "Rejected: illegal (%)": round(kinds.get("illegal", 0), 1),
            "Rejected: unparseable (%)": round(kinds.get("unparseable", 0), 1),
            "Rejected: legal, loses 200+ cp (%)": round(kinds.get("blunder", 0), 1),
            "Median loss of legal rejected moves (cp)": legal.median() if len(legal) else None}


rows = {f"{DPO_RUN} (mistakes of {START})": pair_summary(pairs, answered)}
v2_file = REPO_ROOT / "data" / "processed" / "dpo_pairs_sft_mistakes.jsonl"
if v2_file.exists():
    v2 = pd.read_json(v2_file, lines=True)
    rows["dpo_v2 / dpo_v3 (mistakes of sft_v1)"] = pair_summary(v2, int(v2.positions_answered.iloc[0]))
pair_table = pd.DataFrame(rows).T
pair_table.index.name = "Pairs"
save_table(pair_table, FIGURES, "pairs", "Rejected answers in the preference pairs. cp = centipawns (100 = one pawn).")
display(pair_table)

## 2. SFT → DPO (`sft_legal_dpo`)

DPO with `dpo_v3`'s setting, starting from `sft_legal_moves`, which is also the frozen reference model. The prompts are longer with the list, so the maximum training length is set above the longest pair (as in notebook 13). The figures are the same as in notebook 09:
- **training loss**;
- **DPO diagnostics.** Left: how often the model prefers the chosen answer over the rejected one (50% = no preference). Right: the probability of the chosen and rejected answers compared with the starting model (1× = unchanged). With the SFT loss, the chosen line should stay near 1× while the rejected line falls.

In [ ]:
if is_done(DPO_RUN):
    print(f"{DPO_RUN}: already trained and graded (set REDO = True to train again)")
else:
    tokenizer = load_tokenizer(sft_cfg["model"])
    def length(row, answer):
        text = tokenizer.apply_chat_template(row["prompt"] + answer, tokenize=False)
        return len(tokenizer(text, add_special_tokens=False).input_ids)
    longest = max(max(length(r, r["chosen"]), length(r, r["rejected"])) for r in pairs.to_dict("records"))
    dpo_cfg["training"]["max_seq_length"] = max(256, math.ceil(longest / 64) * 64)
    del tokenizer
    print(f"Longest pair: {longest} tokens -> maximum training length {dpo_cfg['training']['max_seq_length']}")

    train_ds, val_ds = split_pairs(pairs, dpo_cfg["training"]["val_fraction"], dpo_cfg["training"]["seed"])
    print(f"train: {len(train_ds):,} pairs  held out: {len(val_ds):,} pairs")
    log, compute = train_dpo(train_ds, val_ds, dpo_cfg, sft_cfg["model"], sft_cfg["lora"], START_ADAPTER,
                             adapter_dir(DPO_RUN))
    save_training(DPO_RUN, {"dpo": dpo_cfg, "prompt": prompt_cfg}, compute, log, start_adapter=START_ADAPTER,
                  extra={"pairs_file": PAIRS_FILE.name, "pairs": len(pairs)})
    plots.loss_curve(log, FIGURES / f"{DPO_RUN}_training_loss.png", title=f"{DPO_RUN}: loss during training")
    plots.dpo_diagnostics(log, dpo_cfg["training"]["beta"], FIGURES / f"{DPO_RUN}_dpo_diagnostics.png",
                          title=f"{DPO_RUN}: does the model prefer Stockfish's move over its own mistake?",
                          rejected_label="rejected (the model's own mistake)")
    plt.show()
    grade(DPO_RUN)

## 3. SFT → RL (`sft_legal_rl`) and SFT → DPO → RL (`sft_legal_dpo_rl`)

RL with `grpo_v5`'s setting on the same 2,000 training positions as every earlier RL run, now with the legal-move prompt. `sft_legal_rl` starts from `sft_legal_moves`; `sft_legal_dpo_rl` starts from `sft_legal_dpo`. In both, the KL penalty is measured against the model the run starts from.

For each run: the average reward of the sampled answers and the KL divergence from the starting model during training. With the list, almost every sampled answer should be legal, so the reward now mostly measures how many centipawns each move loses.

In [ ]:
t = rl_cfg["training"]
rl_ds = to_rl_dataset(train_positions.sample(n=t["num_positions"], random_state=t["seed"]), prompt_cfg)
print(f"{len(rl_ds):,} positions x {t['num_generations']} answers each = {len(rl_ds) * t['num_generations']:,} "
      "answers scored by Stockfish\n")

for run, start in [(RL_RUN, START), (DPO_RL_RUN, DPO_RUN)]:
    if is_done(run):
        print(f"{run}: already trained and graded (set REDO = True to train again)")
        continue
    start_adapter = START_ADAPTER if start == START else adapter_dir(start)
    print(f"=== {run}: RL from {start} | beta {t['beta']} | reward {rl_cfg['reward']} ===")
    log, compute = train_rl(rl_ds, rl_cfg, eval_cfg["stockfish"], sft_cfg["model"], sft_cfg["lora"], start_adapter,
                            adapter_dir(run))
    save_training(run, {"rl": rl_cfg, "prompt": prompt_cfg}, compute, log, start_adapter=start_adapter)
    rewards = log.reward.dropna()
    print(f"Average reward: {rewards.iloc[0]:.3f} at the start -> {rewards.iloc[-1]:.3f} at the end")
    plots.reward_curve(log, FIGURES / f"{run}_reward.png", title=f"{run}: average reward of the sampled answers")
    if "kl" in log:
        plots.training_curve(log, "kl", f"KL divergence from {start}", FIGURES / f"{run}_kl.png",
                             title=f"{run}: how far the model moves from {start}")
    plt.show()
    grade(run)

## 4. Results: does combining the methods help?

**How to read this section**
- **The main comparison is each combination vs. `sft_legal_moves`**, the model they all start from, with the same prompt. A combination helps if it solves more puzzles than `sft_legal_moves`. (In notebook 11, it should also lose fewer centipawns on game positions without playing Stockfish's move less often.)
- `sft_v1`, `dpo_v3` and `grpo_v5` are shown for reference: the same DPO and RL settings applied to the first SFT model, without the list. They show whether DPO and RL behave differently once legality is solved.
- **The paired comparisons** count puzzles solved by only one of two models; a p-value below 0.05 means the difference is unlikely to be chance.
- **What kind of moves:** the share of pawn moves, captures and checks among legal answers. Earlier RL runs shifted towards quiet pawn moves; puzzles need captures and checks.
- **The training curves** put both RL runs (and `grpo_v5`, without the list) on one figure.

All numbers are on the 1,000 test puzzles; a random legal move solves 5.6%.

In [ ]:
combos = [DPO_RUN, RL_RUN, DPO_RL_RUN]
graded = set(pd.read_csv(experiments.RESULTS / "summary.csv").run)
runs = [run for run in ["sft_v1", "dpo_v3", "grpo_v5", START] + combos if run in graded]
table = puzzle_table(runs)
table.insert(0, "Legal moves in the prompt", ["yes" if run in [START] + combos else "no" for run in table.index])
save_table(table, FIGURES, "puzzle_results", "1,000 test puzzles (250 per rating band). Rating-band columns are "
                                             "puzzle accuracy (%) in that band. ACPL is over legal answers only.")
display(table)

pairs_to_compare = [(DPO_RUN, START), (RL_RUN, START), (DPO_RL_RUN, START), (DPO_RL_RUN, DPO_RUN), (DPO_RL_RUN, RL_RUN),
                    (START, "sft_v1")]
comparison = pd.DataFrame([compare_with(new, old) for new, old in pairs_to_compare
                           if new in graded and old in graded]).set_index("model")
save_table(comparison, FIGURES, "paired_comparison",
           "Puzzles solved by only one of the two models, and the sign-test p-value (below 0.05 = unlikely to be chance).")
display(comparison)

In [ ]:
puzzles = pd.read_csv(REPO_ROOT / "data" / "eval" / "puzzles_test.csv").set_index("puzzle_id")
kinds = pd.DataFrame({run: move_kinds(g, puzzles.fen.loc[g.puzzle_id]) for run, g in
                      ((run, load_graded(run)) for run in runs)}).T.round(1)
kinds.columns = ["Pawn moves (%)", "Captures (%)", "Checks (%)"]
kinds.index.name = "Model"
save_table(kinds, FIGURES, "move_kinds", "Share of each model's legal test answers that are pawn moves, captures, or checks.")
display(kinds)

summary = pd.read_csv(experiments.RESULTS / "summary.csv")
plots.compare_metrics(summary, runs, FIGURES / "puzzle_metrics.png", title="The combinations and their starting models");
plots.accuracy_by_band(summary, runs, FIGURES / "accuracy_by_band.png", title="Puzzle accuracy by puzzle difficulty");
plots.outcome_breakdown(summary, runs, FIGURES / "answer_outcomes.png", title="Are the answers legal moves?");
plots.metric_panels(kinds, [(c, c, "{:.0f}") for c in kinds.columns], FIGURES / "move_kinds.png",
                    title="What kind of moves does each model play? (legal test answers)");

rl_runs = [run for run in ["grpo_v5", RL_RUN, DPO_RL_RUN] if run in graded]
logs = {run if run != "grpo_v5" else "grpo_v5 (no list)": load_training_log(run) for run in rl_runs}
plots.training_curves(logs, "reward", "average reward of sampled answers", FIGURES / "rl_reward.png",
                      title="RL: average reward during training");
logs_with_kl = {name: log for name, log in logs.items() if "kl" in log}
if logs_with_kl:
    plots.training_curves(logs_with_kl, "kl", "KL divergence from the starting model", FIGURES / "rl_kl.png",
                          title="RL: how far each model moves from the model it started from");

## Summary of outputs

- `results/figures/combinations/`: `pairs` (table: what the rejected answers are), `sft_legal_dpo_training_loss.png`, `sft_legal_dpo_dpo_diagnostics.png`, `sft_legal_rl_reward.png`, `sft_legal_rl_kl.png`, `sft_legal_dpo_rl_reward.png`, `sft_legal_dpo_rl_kl.png`, `puzzle_results` (table), `paired_comparison` (table), `move_kinds` (table and figure), `puzzle_metrics.png`, `accuracy_by_band.png`, `answer_outcomes.png`, `rl_reward.png`, `rl_kl.png`
- `results/sft_legal_dpo.json`, `results/sft_legal_rl.json`, `results/sft_legal_dpo_rl.json`: every test answer and its grade; `*_training.json`: settings, compute, training log
- All three models are also graded on held-out game positions and in full games in notebook 11, with the legal moves in the prompt (rerun notebook 11 after this one).